In [0]:
# XGBoost — the machine learning model we'll use for churn prediction
# SHAP — explains why the model made each prediction (tells us which factors matter most for churn)
%pip install xgboost shap

In [0]:
# NOTEBOOK 5 — CHURN PREDICTION
# Using RFM data and customer behaviour to predict which customers are likely to churn
# --------------------------------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import shap
import warnings
warnings.filterwarnings('ignore')

# Load RFM table — contains segments and scores
rfm = pd.read_parquet("/Volumes/workspace/default/retail_data/rfm.parquet")

# Load profiling dataset — contains enriched transaction data
df_profiling = pd.read_parquet("/Volumes/workspace/default/retail_data/df_profiling.parquet")

print(f"Data loaded successfully")
print(f"RFM table: {len(rfm):,} customers")
print(f"Profiling dataset: {len(df_profiling):,} rows")
print(f"\nSegment distribution:")
print(rfm['Segment'].value_counts())

In [0]:
# DEFINING CHURN
# Churn = a customer who has not purchased
# in the last 180 days (6 months)
#
# Why 180 days?
# Our data shows average recency is 200 daysvand median is 95 days. A customer inactive for 6+ months is significantly disengaged compared to the active customer base.
# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Define churn threshold
CHURN_DAYS = 180

# Label each customer as churned or not
# 1 = churned (inactive for 180+ days)
# 0 = active (purchased within 180 days)
rfm['Churned'] = (rfm['Recency'] >= CHURN_DAYS).astype(int)

# Summary
churned = rfm['Churned'].sum()
active = len(rfm) - churned
churn_rate = (churned / len(rfm) * 100).round(2)

print("CHURN DEFINITION")
print(f"Churn threshold:     {CHURN_DAYS} days")
print(f"Active customers:    {active:,} ({100 - churn_rate}%)")
print(f"Churned customers:   {churned:,} ({churn_rate}%)")
print(f"\nChurn rate: {churn_rate}%")

# Visualise churn distribution
plt.figure(figsize=(10, 5))
bars = plt.bar(
    ['Active', 'Churned'],
    [active, churned],
    color=['#2ecc71', '#e74c3c']
)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 20,
        f'{int(bar.get_height()):,}',
        ha='center',
        fontsize=12,
        fontweight='bold'
    )

plt.title('Active vs Churned Customers', fontsize=16, fontweight='bold')
plt.ylabel('Number of Customers', fontsize=12)
plt.tight_layout()
plt.show()

In [0]:
# FEATURE ENGINEERING
# Creating the input variables (features) that XGBoost will use to predict churn

# Features are the clues we give the model to help it identify churned customers

# XGBoost — accurate ML model that learns patterns from data to predict which customers will churn

# SHAP — explains why the model made each prediction so non-technical stakeholders can understand it
# ---------------------------------------------------------------------------------------------------------

# Start with RFM scores as base features
features = rfm[[
    'Customer ID',
    'Recency',
    'Frequency',
    'TotalSpend',
    'R_Score',
    'F_Score',
    'S_Score',
    'Total_Score',
    'Churned'
]].copy()

# Add average order value per customer
avg_order = df_profiling.groupby('Customer ID')['TotalRevenue'].mean().reset_index()
avg_order.columns = ['Customer ID', 'Avg_Order_Value']
features = features.merge(avg_order, on='Customer ID', how='left')

# Add unique products purchased per customer
unique_products = df_profiling.groupby('Customer ID')['Description'].nunique().reset_index()
unique_products.columns = ['Customer ID', 'Unique_Products']
features = features.merge(unique_products, on='Customer ID', how='left')

# Add unique countries per customer
# A customer ordering from multiple countries
# suggests a business buyer not an individual
unique_countries = df_profiling.groupby('Customer ID')['Country'].nunique().reset_index()
unique_countries.columns = ['Customer ID', 'Unique_Countries']
features = features.merge(unique_countries, on='Customer ID', how='left')

# Add average quantity per order
avg_quantity = df_profiling.groupby('Customer ID')['Quantity'].mean().reset_index()
avg_quantity.columns = ['Customer ID', 'Avg_Quantity']
features = features.merge(avg_quantity, on='Customer ID', how='left')

# Add number of days between first and last purchase
# Shows how long the customer has been active
customer_dates = df_profiling.groupby('Customer ID')['InvoiceDate'].agg(
    ['min', 'max']
).reset_index()
customer_dates.columns = ['Customer ID', 'First_Purchase', 'Last_Purchase']
customer_dates['Customer_Lifespan'] = (
    customer_dates['Last_Purchase'] - customer_dates['First_Purchase']
).dt.days
features = features.merge(
    customer_dates[['Customer ID', 'Customer_Lifespan']],
    on='Customer ID',
    how='left'
)

print("FEATURE SUMMARY")
print(f"Total customers: {len(features):,}")
print(f"Total features: {len(features.columns) - 2}")  # exclude Customer ID and Churned
print(f"\nFeature columns:")
for col in features.columns:
    if col not in ['Customer ID', 'Churned']:
        print(f"  → {col}")

print(f"\nMissing values:")
print(features.isnull().sum())

print("\nSAMPLE OF FEATURE TABLE")
print(features.head(5).to_string(index=False))

In [0]:
# TRAIN XGBOOST CHURN PREDICTION MODEL
# We split data into training and testing sets
# Train the model on 80% of customers
# Test it on the remaining 20% it has never seen
# -----------------------------------------------

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Define features (X) and target (y)
# X = the 12 clues we give the model
# y = what we want it to predict (churned or not)

# ORIGINAL CODE — removed due to data leakage
# Recency directly defines churn (Recency >= 180)
# including it as a feature gives the model the answer. The model figured out that every time Recency >= 180 the answer is 1. Every time Recency < 180 the answer is 0.

# X = features.drop(columns=['Customer ID', 'Churned'])

# Removed Recency and RFM scores derived from it
X = features.drop(columns=[
    'Customer ID',
    'Churned',
    'Recency',
    'R_Score',
    'Total_Score'
])



y = features['Churned']

# Split into training and testing sets
# 80% training — model learns from this
# 20% testing — we test on data model has never seen
# random_state=42 ensures we get the same split every time
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set: {len(X_train):,} customers")
print(f"Testing set:  {len(X_test):,} customers")

# Train the XGBoost model
model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

print(f"\nModel trained successfully")
print(f"Features used: {X.columns.tolist()}")

In [0]:
# MODEL EVALUATION
# Testing how well the model predicts churn on the 20% of customers it has never seen
# --------------------------------------------------------------------------------------

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Make predictions on test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# ROC AUC Score — overall model performance
# Score between 0 and 1 — closer to 1 is better
# 0.5 = random guessing, 1.0 = perfect predictions
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"MODEL PERFORMANCE")
print(f"ROC AUC Score: {auc_score:.4f}")

# Classification Report
# Shows precision, recall and f1 score
print(f"\nCLASSIFICATION REPORT")
print(classification_report(y_test, y_pred,
    target_names=['Active', 'Churned']))

# Confusion Matrix
print(f"CONFUSION MATRIX")
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Active', 'Churned'],
    yticklabels=['Active', 'Churned']
)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()


# Comparing default threshold (0.5) vs adjusted threshold (0.3)
# Lower threshold catches more churning customers — better for retention
# Comparing default 0.5 vs adjusted 0.3

# Tested thresholds: 0.2, 0.25, 0.3, 0.35
# 0.3 chosen as optimal balance between catching
# churners and minimising unnecessary false positives
# ------------------------------------------------------------------------------

threshold = 0.3
y_pred_adjusted = (y_pred_proba >= threshold).astype(int)

print(f"\nADJUSTED THRESHOLD: {threshold}")
print(classification_report(y_test, y_pred_adjusted,
    target_names=['Active', 'Churned']))

# Plot both confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Original threshold 0.5
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Active', 'Churned'],
    yticklabels=['Active', 'Churned'],
    ax=axes[0])
axes[0].set_title('Default Threshold (0.5)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Adjusted threshold 0.3
cm_adjusted = confusion_matrix(y_test, y_pred_adjusted)
sns.heatmap(cm_adjusted, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Active', 'Churned'],
    yticklabels=['Active', 'Churned'],
    ax=axes[1])
axes[1].set_title('Adjusted Threshold (0.3)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.suptitle('Confusion Matrix Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

fn = cm_adjusted[1][0]
fp = cm_adjusted[0][1]

print(f"\nOriginal threshold (0.5):")
print(f"Missed churners:     102")
print(f"Wrong churn flags:   209")
print(f"\nAdjusted threshold ({threshold}):")
print(f"Missed churners:     {fn}")
print(f"Wrong churn flags:   {fp}")
print(f"\nCaught {102 - fn} more churning customers")
print(f"{fp - 209} more active customers incorrectly flagged")